# Notebook 02 — Vector Index: Flat Embeddings for RAG
**Layer:** RAG Foundation  
**Embedding model:** `gemma3` via local ollama — no API key, no internet  
**Inputs:** `hdb_feature_table_20260403.csv`  
**Outputs:** `03_vector_index/index_pre2023.faiss` · `index_post2023.faiss` · `embed_config.json`

## Setup — run once before this notebook
```bash
# Install ollama: https://ollama.com
ollama pull gemma3
```

## 2.1 Imports and constants

In [3]:
import pandas as pd
import numpy as np
import faiss
import json
from pathlib import Path
from sklearn.preprocessing import normalize
from tqdm import tqdm
import ollama

ROOT = Path('D:\Master Degree\Projects\Transparent_AI\PropertyLens')
FEATURE_DIR = ROOT / '02_feature_layer' / 'training' / 'outputs'
CSV_PATH    = FEATURE_DIR / "hdb_feature_table_20260412.csv"
INDEX_DIR = Path("03_vector_index")
INDEX_DIR.mkdir(exist_ok=True)

OLLAMA_MODEL        = "nomic-embed-text"   # local ollama model
EMBED_BATCH_SIZE    = 256         # gemma3 is large — smaller batches avoid OOM
TEMPORAL_SPLIT_YEAR = 2023

# Probe embedding dimension from a test call
test_resp = ollama.embed(model=OLLAMA_MODEL, input=["test"])
EMBED_DIM = len(test_resp.embeddings[0])
print(f"ollama model    : {OLLAMA_MODEL}")
print(f"embedding dim   : {EMBED_DIM}")

ollama model    : nomic-embed-text
embedding dim   : 768


## 2.2 Load CSV and decode one-hot columns

In [4]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Pre-2023  : {(df.transaction_year < TEMPORAL_SPLIT_YEAR).sum():,}")
print(f"Post-2023 : {(df.transaction_year >= TEMPORAL_SPLIT_YEAR).sum():,}")

TOWN_COLS  = [c for c in df.columns if c.startswith("town_")]
TYPE_COLS  = [c for c in df.columns if c.startswith("flat_type_")]
MODEL_COLS = [c for c in df.columns if c.startswith("flat_model_")]

def decode_onehot(row, cols, prefix):
    for c in cols:
        if row[c] == 1: return c.replace(prefix, "")
    return "UNKNOWN"

df["town"]       = df.apply(lambda r: decode_onehot(r, TOWN_COLS,  "town_"),       axis=1)
df["flat_type"]  = df.apply(lambda r: decode_onehot(r, TYPE_COLS,  "flat_type_"),  axis=1)
df["flat_model"] = df.apply(lambda r: decode_onehot(r, MODEL_COLS, "flat_model_"), axis=1)
print("Decoded.")

Loaded: 260,699 rows x 73 cols
Pre-2023  : 178,589
Post-2023 : 82,110
Decoded.


## 2.3 Build embedding text per flat

In [5]:
def build_embed_text(row):
    return (
        f"{row['town']} {row['flat_type']} {row['flat_model']} "
        f"storey={row['level_mid']:.1f} "
        f"area={row['floor_area_sqm']:.1f}sqm "
        f"rooms={int(row['room_count'])} "
        f"lease={row['lease_remaining_years']:.0f}yrs "
        f"price={int(row['resale_price'])} "
        f"year={int(row['transaction_year'])} "
        f"mrt={row['dist_to_mrt_m']:.0f}m "
        f"school={row['dist_to_nearest_school_m']:.0f}m "
        f"mall_count={row['mall_count_3km']:.0f} "
        f"school_quality={row['primary_school_quality_1km_weighted']:.2f}"
    )

df["embed_text"] = df.apply(build_embed_text, axis=1)
print("Sample:", df["embed_text"].iloc[0])
print(f"Total texts: {len(df):,}")

Sample: ANG MO KIO 3 ROOM Improved storey=8.0 area=60.0sqm rooms=3 lease=70yrs price=255000 year=2015 mrt=1176m school=415m mall_count=5 school_quality=14.13
Total texts: 260,699


## 2.4 Generate embeddings via ollama gemma3

2.4.1 Generate embeddings via Gemini Genai

In [6]:
import google.generativeai as genai
import os
from dotenv import load_dotenv
ENV_PATH = ROOT / '.env'
load_dotenv(dotenv_path=ENV_PATH)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

for m in genai.list_models():
    if "embed" in m.name.lower():
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview


In [ ]:

texts = df["embed_text"].tolist()
n     = len(texts)
all_vecs = np.zeros((n, EMBED_DIM), dtype=np.float32)

for i in tqdm(range(0, n, EMBED_BATCH_SIZE), desc="Embedding via ollama"):
    batch = texts[i : i + EMBED_BATCH_SIZE]
    resp  = ollama.embed(model=OLLAMA_MODEL, input=batch)
    vecs  = np.array(resp.embeddings, dtype=np.float32)
    all_vecs[i : i + len(vecs)] = vecs

# L2 normalise for cosine similarity via inner product
all_vecs = normalize(all_vecs, norm="l2").astype(np.float32)
print(f"Embedding matrix: {all_vecs.shape}")

#json.dump({"embed_dim": EMBED_DIM, "method": "ollama_gemma3"},
json.dump({"embed_dim": EMBED_DIM, "method": "nomic-embed-text"},
          open(INDEX_DIR / "embed_config.json", "w"), indent=2)
np.save(INDEX_DIR / "all_embeddings.npy", all_vecs)
print("Saved: embed_config.json, all_embeddings.npy")

Embedding via ollama:   0%|          | 0/2607 [00:00<?, ?it/s]

Embedding via ollama: 100%|██████████| 2607/2607 [34:03<00:00,  1.28it/s] 


Embedding matrix: (260699, 768)
Saved: embed_config.json, all_embeddings.npy


## 2.5 Build FAISS indexes with temporal split

In [9]:
META_COLS = [
    "address_key","resale_price","transaction_year",
    "town","flat_type","flat_model","floor_area_sqm",
    "level_mid","lease_remaining_years","room_count",
    "dist_to_nearest_school_m","mall_count_3km",
    "primary_school_quality_1km_weighted",
]
meta_df = df[META_COLS].copy()

for split_name, mask in [
    ("pre2023",  df["transaction_year"] <  TEMPORAL_SPLIT_YEAR),
    ("post2023", df["transaction_year"] >= TEMPORAL_SPLIT_YEAR),
]:
    idx  = np.where(mask)[0]
    vecs = all_vecs[idx].copy()
    faiss.normalize_L2(vecs)
    index = faiss.IndexFlatIP(EMBED_DIM)
    index.add(vecs)
    faiss.write_index(index, str(INDEX_DIR / f"index_{split_name}.faiss"))
    meta_df.iloc[idx].to_parquet(INDEX_DIR / f"meta_{split_name}.parquet", index=False)
    mean_p = meta_df.iloc[idx]["resale_price"].mean()
    print(f"{split_name}: {index.ntotal:,} vectors  mean price ${mean_p:,.0f}")

print("\nNote: 31% price drift between slices is expected (README section 3).")
print("Notebook 02 complete.")

pre2023: 178,589 vectors  mean price $468,788
post2023: 82,110 vectors  mean price $614,711

Note: 31% price drift between slices is expected (README section 3).
Notebook 02 complete.
